In [ ]:
import requests
import os
from dotenv import load_dotenv
import json
import pandas as pd
import datetime
import time

EXTRACT

In [ ]:
#loading API key from secrets/.env
load_dotenv()
API_KEY =  os.getenv("API_KEY")

In [ ]:
base_url = "https://backend.simfin.com/api/v3/companies"

endpoint_url= f"{base_url}/prices/compact"

headers = {"accept": "application/json",
            "Authorization": f"{API_KEY}"}

In [ ]:
#Creating PySimFin class. This is the API wrapper we will use to make API calls.

class PySimFin:
    def __init__(self):
        self.endpoint_url = f"{endpoint_url}"
        self.headers = headers.copy()   

    def get_share_prices(self, ticker: str):
        params = {'ticker': ticker}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")

    def get_share_prices_verbose(self, ticker: str, start: str, end: str):
        params = {'ticker': ticker,
                  'start': start,
                  'end': end}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")
    

    def get_share_prices_today(self, ticker: str):
        today = str(datetime.datetime.today()).split()[0] #gets today's date
        params = {'ticker': ticker,
                  'start': today}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")

In [ ]:
# This will enventually need to go to the config.yaml
top_40_companies =[
{"name" : "Nvidia","ticker": "NVDA"},
{"name" : "Apple Inc.","ticker": "AAPL"},
{"name" : "Microsoft",	"ticker": "MSFT"},
{"name" : "Amazon",	"ticker": "AMZN"},
{"name" : "Meta Platforms",	"ticker": "META"},	
{"name" : "Alphabet Inc. (Class C)","ticker":  "GOOG"},	
{"name" : "Tesla, Inc.", "ticker": "TSLA"},
{"name" : "Walmart", "ticker": "WMT"},
{"name" : "JPMorgan Chase",	"ticker": "JPM"},	
{"name" : "Oracle Corporation",	"ticker": "ORCL"},	
{"name" : "Visa Inc.", "ticker": "V"},
{"name" : "Mastercard",	"ticker": "MA"},
{"name" : "Netflix", "ticker": "NFLX"},	
{"name" : "ExxonMobil",	"ticker": "XOM"},	
{"name" : "Johnson & Johnson", "ticker": "JNJ"},	
{"name" : "Palantir Technologies", "ticker": "PLTR"},	
{"name" : "Costco",	"ticker": "COST"},	
{"name" : "Home Depot",	"ticker": "HD"},	
{"name" : "Bank of America", "ticker": "BAC"},	
{"name" : "Procter & Gamble", "ticker": "PG"},	
{"name" : "Chevron Corporation", "ticker": "CVX"},	
{"name" : "Coca-Cola Company", "ticker": "KO"},	
{"name" : "Cisco", "ticker": "CSCO"},	
{"name" : "Wells Fargo","ticker": "WFC"},
{"name" : "IBM", "ticker": "IBM"},	
{"name" : "T-Mobile US", "ticker": "TMUS"},	
{"name" : "Morgan Stanley",	"ticker": "MS"},	
{"name" : "Salesforce",	"ticker": "CRM"},	
{"name" : "Caterpillar Inc.","ticker": "CAT"},
{"name" : "American Express", "ticker": "AXP"},
{"name" : "Philip Morris International","ticker":  "PM"},	
{"name" : "Goldman Sachs", "ticker": "GS"},	
{"name" : "RTX Corporation", "ticker": "RTX"},	
{"name" : "Abbott Laboratories", "ticker": "ABT"},
{"name" : "McDonald's",	"ticker": "MCD"},	
{"name" : "PepsiCo", "ticker": "PEP"},
{"name" : "Walt Disney Company","ticker":  "DIS"},	
{"name" : "ServiceNow",	"ticker": "NOW"},
{"name" : "AT&T", "ticker": "T"},
{"name" : "Intel", "ticker": "INTC"},
]

In [ ]:
# Calling API for top 40 Companies

stock_price_today = PySimFin()
price_data = []

for company in top_40_companies:
    print(company['ticker']) #Can get rid of this later on
    ticker = str(company['ticker'])
    #ind_stock_price = stock_price_today.get_share_prices_today(ticker) #Should use this one --> issue is that on weekends it will be empty
    ind_stock_price = stock_price_today.get_share_prices_verbose(ticker, '2025-11-07','2025-11-08')
    price_data.extend(ind_stock_price)
    time.sleep(0.5) #Need to pause execution for 0.5 seconds as only 2 requests are alowed per minute on SymFin.

price_data

TRANSFORM

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
#This function is definite. it works well

def transformations(price_data: list):
    #creates columns
    dict_data = price_data[0]
    columns = ['name', 'ticker', 'currency']
    info = dict_data['columns']
    columns.extend(info)
    df = pd.DataFrame(columns=columns)
    
    #Appends the data as rows to the DF
    for i in price_data:
        data = [i['name'], i['ticker'], i['currency']]
        stock_data = i['data'][0]
        print(stock_data)
        data.extend(stock_data)
        print(data)
        df.loc[len(df)] = data

    return df #df here is a local variable

In [ ]:
df =transformations(price_data) #here we create the df to be able to save it
df
#Need to fix in case of all null values for a ticker

In [ ]:
df.dtypes

LOAD

In [ ]:
#Only write to csv for now. Later on we will send to Postgres and PowerBI
df.to_csv('data/output.csv', index=False)

trying out config

In [ ]:
import yaml
from pathlib import Path

In [ ]:
def load_config(config_path: str = "config.yaml") -> dict:
    """
    Loads config file to centralize and control project behavior.

    Why: Externalizing settings avoids hardcoding parameters across modules,
    supports easier collaboration, and allows quick updates or environment
    switches without touching the core logic.
    """
    if not os.path.isfile(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")
    with open(config_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    return config

In [ ]:
config = load_config('config.yaml')

In [ ]:
api_cfg = config.get('api_information')['headers']
api_cfg

In [ ]:
#   NEED TO DO HEADERS NOW, WITH API_KEY

In [ ]:
#constructing endpoint_url --> use in script.
base_url= config.get('api_information')['base_url']
endpoint = config.get('api_information')['endpoint_url']
endpoint_url = f"{base_url}{endpoint_url}"
endpoint_url

In [ ]:
#constructing header   use in script.
headers = config.get('api_information')['headers']
headers['Authorization'] = f"{API_KEY}"


In [ ]:
#Creating PySimFin class. This is the API wrapper we will use to make API calls.

class PySimFin:
    def __init__(self):
        self.endpoint_url = f"{endpoint_url}"
        self.headers = headers.copy()   

    def get_share_prices(self, ticker: str):
        params = {'ticker': ticker}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")

    def get_share_prices_verbose(self, ticker: str, start: str, end: str):
        params = {'ticker': ticker,
                  'start': start,
                  'end': end}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")
    

    def get_share_prices_today(self, ticker: str):
        today = str(datetime.datetime.today()).split()[0] #gets today's date
        params = {'ticker': ticker,
                  'start': today}
        response = requests.get(self.endpoint_url, headers=self.headers, params=params)
        if response.status_code == 200:
            data = response.json()
            return data
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")

In [ ]:
base_url = "https://backend.simfin.com/api/v3/companies"

endpoint_url= f"{base_url}/prices/compact"

headers = {"accept": "application/json",
            "Authorization": f"{API_KEY}"}

In [ ]:
# This will enventually need to go to the config.yaml
top_40_companies =[
{"name" : "Nvidia","ticker": "NVDA"},
{"name" : "Apple Inc.","ticker": "AAPL"},
{"name" : "Microsoft",	"ticker": "MSFT"},
{"name" : "Amazon",	"ticker": "AMZN"},
{"name" : "Meta Platforms",	"ticker": "META"},	
{"name" : "Alphabet Inc. (Class C)","ticker":  "GOOG"},	
{"name" : "Tesla, Inc.", "ticker": "TSLA"},
{"name" : "Walmart", "ticker": "WMT"},
{"name" : "JPMorgan Chase",	"ticker": "JPM"},	
{"name" : "Oracle Corporation",	"ticker": "ORCL"},	
{"name" : "Visa Inc.", "ticker": "V"},
{"name" : "Mastercard",	"ticker": "MA"},
{"name" : "Netflix", "ticker": "NFLX"},	
{"name" : "ExxonMobil",	"ticker": "XOM"},	
{"name" : "Johnson & Johnson", "ticker": "JNJ"},	
{"name" : "Palantir Technologies", "ticker": "PLTR"},	
{"name" : "Costco",	"ticker": "COST"},	
{"name" : "Home Depot",	"ticker": "HD"},	
{"name" : "Bank of America", "ticker": "BAC"},	
{"name" : "Procter & Gamble", "ticker": "PG"},	
{"name" : "Chevron Corporation", "ticker": "CVX"},	
{"name" : "Coca-Cola Company", "ticker": "KO"},	
{"name" : "Cisco", "ticker": "CSCO"},	
{"name" : "Wells Fargo","ticker": "WFC"},
{"name" : "IBM", "ticker": "IBM"},	
{"name" : "T-Mobile US", "ticker": "TMUS"},	
{"name" : "Morgan Stanley",	"ticker": "MS"},	
{"name" : "Salesforce",	"ticker": "CRM"},	
{"name" : "Caterpillar Inc.","ticker": "CAT"},
{"name" : "American Express", "ticker": "AXP"},
{"name" : "Philip Morris International","ticker":  "PM"},	
{"name" : "Goldman Sachs", "ticker": "GS"},	
{"name" : "RTX Corporation", "ticker": "RTX"},	
{"name" : "Abbott Laboratories", "ticker": "ABT"},
{"name" : "McDonald's",	"ticker": "MCD"},	
{"name" : "PepsiCo", "ticker": "PEP"},
{"name" : "Walt Disney Company","ticker":  "DIS"},	
{"name" : "ServiceNow",	"ticker": "NOW"},
{"name" : "AT&T", "ticker": "T"},
{"name" : "Intel", "ticker": "INTC"},
]

In [ ]:
# Calling API for top 40 Companies

stock_price_today = PySimFin()
price_data = []

for company in top_40_companies:
    print(company['ticker']) #Can get rid of this later on
    ticker = str(company['ticker'])
    ind_stock_price = stock_price_today.get_share_prices_today(ticker) #Should use this one --> issue is that on weekends it will be empty
    #ind_stock_price = stock_price_today.get_share_prices_verbose(ticker, '2025-11-07','2025-11-08')
    price_data.extend(ind_stock_price)
    time.sleep(0.5) #Need to pause execution for 0.5 seconds as only 2 requests are alowed per minute on SymFin.

price_data

Trial and error

In [ ]:
empty_list = []
type(empty_list)

In [ ]:
len(empty_list)

In [ ]:
import json

In [ ]:
# read a json file
with open("data/raw/raw_data.json", "r") as file:
    data = json.load(file)

In [ ]:
data

In [ ]:
data[28]

In [1]:
import os
import yaml
import json

def load_config(config_path: str = "config.yaml") -> dict:
    """
    Loads config file to centralize and control project behavior.

    Why: Externalizing settings avoids hardcoding parameters across modules,
    supports easier collaboration, and allows quick updates or environment
    switches without touching the core logic.
    """
    if not os.path.isfile(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")
    with open(config_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    return config

In [2]:
config = load_config()

In [6]:
def load_raw_data(raw_data_path: json = f"{config.get('data_source')['raw_path']}"):
    with open(f"{config.get('data_source')['raw_path']}", "r") as file:
         price_data = json.load(file)
    return price_data

In [7]:
load_raw_data()

[{'columns': ['Date',
   'Dividend Paid',
   'Common Shares Outstanding',
   'Last Closing Price',
   'Adjusted Closing Price',
   'Highest Price',
   'Lowest Price',
   'Opening Price',
   'Trading Volume'],
  'name': 'NVIDIA CORP',
  'id': 172199,
  'ticker': 'NVDA',
  'currency': 'USD',
  'isin': 'US67066G1040',
  'data': [['2025-11-07',
    None,
    24300000000,
    188.15,
    188.15,
    188.32,
    178.91,
    184.9,
    262255193]]},
 {'columns': ['Date',
   'Dividend Paid',
   'Common Shares Outstanding',
   'Last Closing Price',
   'Adjusted Closing Price',
   'Highest Price',
   'Lowest Price',
   'Opening Price',
   'Trading Volume'],
  'name': 'APPLE INC',
  'id': 111052,
  'ticker': 'AAPL',
  'currency': 'USD',
  'isin': 'US0378331005',
  'data': [['2025-11-07',
    None,
    14776353000,
    268.47,
    268.21,
    272.29,
    266.77,
    269.8,
    48227365]]},
 {'columns': ['Date',
   'Dividend Paid',
   'Common Shares Outstanding',
   'Last Closing Price',
   'Adjust

In [9]:
price_data = load_raw_data()
price_data

[{'columns': ['Date',
   'Dividend Paid',
   'Common Shares Outstanding',
   'Last Closing Price',
   'Adjusted Closing Price',
   'Highest Price',
   'Lowest Price',
   'Opening Price',
   'Trading Volume'],
  'name': 'NVIDIA CORP',
  'id': 172199,
  'ticker': 'NVDA',
  'currency': 'USD',
  'isin': 'US67066G1040',
  'data': [['2025-11-07',
    None,
    24300000000,
    188.15,
    188.15,
    188.32,
    178.91,
    184.9,
    262255193]]},
 {'columns': ['Date',
   'Dividend Paid',
   'Common Shares Outstanding',
   'Last Closing Price',
   'Adjusted Closing Price',
   'Highest Price',
   'Lowest Price',
   'Opening Price',
   'Trading Volume'],
  'name': 'APPLE INC',
  'id': 111052,
  'ticker': 'AAPL',
  'currency': 'USD',
  'isin': 'US0378331005',
  'data': [['2025-11-07',
    None,
    14776353000,
    268.47,
    268.21,
    272.29,
    266.77,
    269.8,
    48227365]]},
 {'columns': ['Date',
   'Dividend Paid',
   'Common Shares Outstanding',
   'Last Closing Price',
   'Adjust

In [12]:
import pandas as pd

In [13]:
#This function is definite. it works well

def transformations(price_data: list):
    #creates columns
    dict_data = price_data[0]
    columns = ['name', 'ticker', 'currency']
    info = dict_data['columns']
    columns.extend(info)
    df = pd.DataFrame(columns=columns)
    
    #Appends the data as rows to the DF
    for i in price_data:
        data = [i['name'], i['ticker'], i['currency']]
        stock_data = i['data'][0]
        print(stock_data)
        data.extend(stock_data)
        print(data)
        df.loc[len(df)] = data

    return df #df here is a local variable

In [14]:
df =transformations(price_data) #here we create the df to be able to save it
df
#Need to fix in case of all null values for a ticker

['2025-11-07', None, 24300000000, 188.15, 188.15, 188.32, 178.91, 184.9, 262255193]
['NVIDIA CORP', 'NVDA', 'USD', '2025-11-07', None, 24300000000, 188.15, 188.15, 188.32, 178.91, 184.9, 262255193]
['2025-11-07', None, 14776353000, 268.47, 268.21, 272.29, 266.77, 269.8, 48227365]
['APPLE INC', 'AAPL', 'USD', '2025-11-07', None, 14776353000, 268.47, 268.21, 272.29, 266.77, 269.8, 48227365]
['2025-11-07', None, 7432377655, 496.82, 496.82, 499.38, 493.25, 496.94, 23857052]
['MICROSOFT CORP', 'MSFT', 'USD', '2025-11-07', None, 7432377655, 496.82, 496.82, 499.38, 493.25, 496.94, 23857052]
['2025-11-07', None, 7432377655, 496.82, 496.82, 499.38, 493.25, 496.94, 23857052]
['MICROSOFT CORP', 'MSFT', 'USD', '2025-11-07', None, 7432377655, 496.82, 496.82, 499.38, 493.25, 496.94, 23857052]
['2025-11-07', None, 2520527597, 621.71, 621.71, 622.13, 601.2, 616.49, 29047168]
['Meta Platforms, Inc.', 'META', 'USD', '2025-11-07', None, 2520527597, 621.71, 621.71, 622.13, 601.2, 616.49, 29047168]
['2025-

C:\Users\trtre\AppData\Local\Temp\ipykernel_5688\1289636035.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df.loc[len(df)] = data
C:\Users\trtre\AppData\Local\Temp\ipykernel_5688\1289636035.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df.loc[len(df)] = data
C:\Users\trtre\AppData\Local\Temp\ipykernel_5688\1289636035.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-

,name,ticker,currency,Date,Dividend Paid,Common Shares Outstanding,Last Closing Price,Adjusted Closing Price,Highest Price,Lowest Price,Opening Price,Trading Volume
0,NVIDIA CORP,NVDA,USD,2025-11-07,NaN,24300000000,188.15,188.15,188.32,178.91,184.90,262255193
1,APPLE INC,AAPL,USD,2025-11-07,NaN,14776353000,268.47,268.21,272.29,266.77,269.80,48227365
2,MICROSOFT CORP,MSFT,USD,2025-11-07,NaN,7432377655,496.82,496.82,499.38,493.25,496.94,23857052
3,MICROSOFT CORP,MSFT,USD,2025-11-07,NaN,7432377655,496.82,496.82,499.38,493.25,496.94,23857052
4,"Meta Platforms, Inc.",META,USD,2025-11-07,NaN,2520527597,621.71,621.71,622.13,601.20,616.49,29047168
5,Alphabet (Google),GOOG,USD,2025-11-07,NaN,12581000000,279.70,279.70,284.50,275.74,284.21,21832162
6,Tesla,TSLA,USD,2025-11-07,NaN,3325819167,429.52,429.52,439.36,421.88,437.92,102578589
7,Walmart Inc,WMT,USD,2025-11-07,NaN,7972851122,102.59,102.59,102.97,101.73,102.13,17212495
8,JPMORGAN CHASE & CO,JPM,USD,2025-11-07,NaN,2722262295,314.21,314.21,314.43,307.64,311.89,7193081
9,ORACLE CORP,ORCL,USD,2025-11-07,NaN,2841714000,239.26,239.26,240.40,232.35,239.00,20232802
